# سیستم شباهت‌یابی مبتنی بر شبکه عصبی (پروژه کارآموزی)

این نوت‌بوک شامل پیاده‌سازی کامل و گام‌به‌گام فعالیت اجرایی کارآموزی به همراه آخرین نیازمندی‌های کارفرما است:
1. **ساخت پایگاه‌داده ۱۰۰ سطری:** هر سطر نماینده یک نمونه داده و دارای ۳ ویژگی برداری بازه‌ای است.
2. **نمایش گرافیکی ۳ بعدی:** مصورسازی توزیع ۱۰۰ سطر در فضای ۳ بعدی ویژگی‌ها و تحلیل ماتریس همپوشانی (سقف ۳۰٪).
3. **تولید داده دریافتی (Query):** مقادیر عددی نقطه‌ای تصادفی با **عدم حضور مستقل ۳۰٪ برای هر یک از اعضا** و ۱٪ نویز.
4. **حل گلوگاه ورودی با طول متغیر در شبکه عصبی:** معماری Padding & Validity Masking بدون سرچ دستی و بدون خلاصه‌سازی آماری.
5. **ارزیابی و رسم نمودار عملکرد:** بررسی دقت آموزش و آزمون و همگرایی تابع هزینه.
6. **ذخیره و بازیابی وزن‌های مدل:** استنتاج سریع با بارگذاری وزن‌ها بدون نیاز به ترین مجدد.


In [ ]:
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib_cache"
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import json
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✅ تمام کتابخانه‌ها با موفقیت بارگذاری شدند.")


## مرحله اول: ساخت پایگاه‌داده ۱۰۰ سطری با ویژگی‌های بازه‌ای

مشخصات بازه‌ها مطابق تعریف کارآموزی:
- **Feature1:** بین ۱ تا ۱۰ بازه، حداکثر عرض بازه ۵، محدوده ۱۰۰۰ تا ۱۰۰۰۰
- **Feature2:** بین ۱ تا ۲۰ بازه، حداکثر عرض بازه ۱۰، محدوده ۲۰۰۰۰ تا ۱۰۰۰۰۰
- **Feature3:** بین ۱ تا ۵ بازه، حداکثر عرض بازه ۱، محدوده ۰.۱ تا ۱۰۰۰
- سقف همپوشانی بازه‌های سطرها با یکدیگر: ۳۰٪


In [ ]:
def generate_intervals(n_min: int, n_max: int, width_max: float, value_min: float, value_max: float, decimal_places: int = 2) -> List[Tuple[float, float]]:
    n = np.random.randint(n_min, n_max + 1)
    intervals = []
    for _ in range(n):
        start = np.random.uniform(value_min, value_max - width_max)
        width = np.random.uniform(0, width_max)
        end = start + width
        start, end = round(start, decimal_places), round(end, decimal_places)
        if start > end:
            start, end = end, start
        intervals.append((start, end))
    intervals.sort(key=lambda x: x[0])
    return intervals

def intervals_to_string(intervals: List[Tuple[float, float]]) -> str:
    return ",".join([f"{s}-{e}" for s, e in intervals])

def generate_database(n_rows: int = 100) -> pd.DataFrame:
    data = {"Feature1": [], "Feature2": [], "Feature3": []}
    for _ in range(n_rows):
        f1 = generate_intervals(1, 10, 5.0, 1000.0, 10000.0, 2)
        f2 = generate_intervals(1, 20, 10.0, 20000.0, 100000.0, 2)
        f3 = generate_intervals(1, 5, 1.0, 0.1, 1000.0, 3)
        data["Feature1"].append(intervals_to_string(f1))
        data["Feature2"].append(intervals_to_string(f2))
        data["Feature3"].append(intervals_to_string(f3))
    df = pd.DataFrame(data)
    df.index.name = "row_id"
    return df

df_database = generate_database(100)
df_database.to_pickle("database.pkl")
df_database.to_csv("database.csv", encoding="utf-8-sig")

print(f"✅ پایگاه‌داده ۱۰۰ سطری با موفقیت ایجاد و ذخیره شد. (تعداد سطرها: {len(df_database)})")
df_database.head()


## مصورسازی گرافیکی ۳ بعدی پایگاه‌داده و ماتریس همپوشانی اطلاعات

مطابق با درخواست کارفرما، توزیع ۱۰۰ سطر پایگاه‌داده در فضای ۳ بعدی $(Feature_1, Feature_2, Feature_3)$ به همراه خطوط ارتباطی همپوشانی و ماتریس گرمایی همپوشانی دوبه‌دو رسم می‌شود:


In [ ]:
def parse_intervals(interval_str: str) -> List[Tuple[float, float]]:
    if not interval_str or pd.isna(interval_str):
        return []
    intervals = []
    for part in str(interval_str).split(","):
        part = part.strip()
        if "-" in part:
            try:
                s, e = part.split("-")
                intervals.append((float(s), float(e)))
            except ValueError:
                continue
    return intervals

def calc_overlap_ratio(ints1: List[Tuple[float, float]], ints2: List[Tuple[float, float]]) -> float:
    if not ints1 or not ints2: return 0.0
    def total_len(ints): return sum(e - s for s, e in ints)
    overlap = 0.0
    for s1, e1 in ints1:
        for s2, e2 in ints2:
            os = max(s1, s2)
            oe = min(e1, e2)
            if os < oe: overlap += (oe - os)
    l1, l2 = total_len(ints1), total_len(ints2)
    return overlap / ((l1 + l2) / 2.0) if (l1 + l2) > 0 else 0.0

n_rows = len(df_database)
centers_f1, centers_f2, centers_f3, num_intervals = [], [], [], []
parsed_all = []

for _, row in df_database.iterrows():
    f1, f2, f3 = parse_intervals(row["Feature1"]), parse_intervals(row["Feature2"]), parse_intervals(row["Feature3"])
    parsed_all.append((f1, f2, f3))
    centers_f1.append(np.mean([(s+e)/2.0 for s, e in f1]) if f1 else 0.0)
    centers_f2.append(np.mean([(s+e)/2.0 for s, e in f2]) if f2 else 0.0)
    centers_f3.append(np.mean([(s+e)/2.0 for s, e in f3]) if f3 else 0.0)
    num_intervals.append(len(f1) + len(f2) + len(f3))

centers_f1, centers_f2, centers_f3 = np.array(centers_f1), np.array(centers_f2), np.array(centers_f3)
num_intervals = np.array(num_intervals)

overlap_matrix = np.zeros((n_rows, n_rows))
for i in range(n_rows):
    for j in range(i + 1, n_rows):
        o1 = calc_overlap_ratio(parsed_all[i][0], parsed_all[j][0])
        o2 = calc_overlap_ratio(parsed_all[i][1], parsed_all[j][1])
        o3 = calc_overlap_ratio(parsed_all[i][2], parsed_all[j][2])
        avg_o = (o1 + o2 + o3) / 3.0
        overlap_matrix[i, j] = avg_o
        overlap_matrix[j, i] = avg_o

fig = plt.figure(figsize=(16, 7))

# ۱. نمودار ۳ بعدی توزیع داده‌ها و یال‌های همپوشانی
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
sc = ax1.scatter(centers_f1, centers_f2, centers_f3, c=np.arange(n_rows), cmap="viridis", s=num_intervals * 7, alpha=0.85, edgecolors="black", linewidth=0.5)

overlap_connections = 0
for i in range(n_rows):
    for j in range(i + 1, n_rows):
        if overlap_matrix[i, j] > 0.03:
            ax1.plot([centers_f1[i], centers_f1[j]], [centers_f2[i], centers_f2[j]], [centers_f3[i], centers_f3[j]], color="red", alpha=min(0.8, overlap_matrix[i, j] * 4.0), linewidth=0.8)
            overlap_connections += 1

ax1.set_xlabel("Feature 1 (1000 - 10000)", fontsize=10, labelpad=8)
ax1.set_ylabel("Feature 2 (20000 - 100000)", fontsize=10, labelpad=8)
ax1.set_zlabel("Feature 3 (0.1 - 1000)", fontsize=10, labelpad=8)
ax1.set_title(f"3D Distribution of 100 Rows & Overlap Graph ({overlap_connections} Connections)", fontsize=11, fontweight="bold")
plt.colorbar(sc, ax=ax1, shrink=0.6, label="Row ID (0 - 99)")

# ۲. ماتریس گرمایی همپوشانی دوبه‌دو
ax2 = fig.add_subplot(1, 2, 2)
im = ax2.imshow(overlap_matrix * 100.0, cmap="magma", aspect="auto", origin="lower")
ax2.set_xlabel("Row Index", fontsize=10)
ax2.set_ylabel("Row Index", fontsize=10)
ax2.set_title(f"Pairwise Overlap Matrix Heatmap (Max Overlap = {overlap_matrix.max()*100:.2f}%)", fontsize=11, fontweight="bold")
plt.colorbar(im, ax=ax2, label="Overlap Percentage (%)")

plt.tight_layout()
plt.show()


## مرحله دوم: تولید داده دریافتی (Query) نقطه‌ای با حذف مستقل ۳۰٪ اعضا

مطابق اصلاحیه جدید کارفرما:
- داده‌های دریافتی به صورت مقادیر عددی (نقاط) از درون بازه‌ها انتخاب می‌شوند.
- **هر یک از اعضا به صورت کاملاً مستقل با احتمال ۳۰٪ می‌تواند مفقود باشد** (Independent Dropout).
- به هر عدد ۱٪ خطای تصادفی اعمال می‌شود.


In [ ]:
def apply_independent_dropout(values: List[float], drop_prob: float = 0.30) -> List[float]:
    kept = [v for v in values if np.random.random() >= drop_prob]
    if not kept and values:
        kept = [values[np.random.randint(0, len(values))]]
    return kept

def generate_feature_points(interval_str: str, decimal_places: int = 4) -> Tuple[List[float], str]:
    intervals = parse_intervals(interval_str)
    if not intervals: return [], ""
    # ۱. انتخاب عدد تصادفی از درون بازه
    points = [round(float(np.random.uniform(s, e)), decimal_places) for s, e in intervals]
    # ۲. اعمال حذف مستقل ۳۰٪ برای هر یک از اعضا
    points = apply_independent_dropout(points, drop_prob=0.30)
    # ۳. اعمال ۱٪ نویز به هر عدد
    noisy_points = [round(float(p * (1.0 + np.random.uniform(-0.01, 0.01))), decimal_places) for p in points]
    points_str = ", ".join([f"{p:.{decimal_places}f}" for p in noisy_points])
    return noisy_points, points_str

def generate_query(df: pd.DataFrame) -> Dict:
    random_idx = int(np.random.randint(0, len(df)))
    row = df.iloc[random_idx]
    f1_pts, f1_str = generate_feature_points(row["Feature1"])
    f2_pts, f2_str = generate_feature_points(row["Feature2"])
    f3_pts, f3_str = generate_feature_points(row["Feature3"])
    return {
        "source_row_id": random_idx,
        "Feature1": f1_str,
        "Feature2": f2_str,
        "Feature3": f3_str,
        "Feature1_values": f1_pts,
        "Feature2_values": f2_pts,
        "Feature3_values": f3_pts,
    }

query_sample = generate_query(df_database)
with open("query.json", "w", encoding="utf-8") as f:
    json.dump(query_sample, f, ensure_ascii=False, indent=2)

print(f"🎯 سطر مبدأ انتخاب‌شده برای کوئری: سطر {query_sample['source_row_id']}")
print(f"Feature1 (اعداد): {query_sample['Feature1']}")
print(f"Feature2 (اعداد): {query_sample['Feature2'][:80]}...")
print(f"Feature3 (اعداد): {query_sample['Feature3']}")


## مرحله سوم: حل گلوگاه ورودی با طول متغیر (Padding & Masking) و آموزش شبکه عصبی

- حل گلوگاه طول متغیر از طریق تانسور ظرفیت ماکزیمم (۳۵ مقدار نرمال‌شده) و بردار ماسک اعتبار (۳۵ مقدار باینری) مجموعاً ۷۰ بعد.
- آموزش شبکه MLP با بهینه‌ساز Adam و Early Stopping.
- رسم نمودار همگرایی تابع هزینه، منحنی اعتبارسنجی و مقایسه دقت آموزش و آزمون.
- ذخیره وزن‌های مدل آموزش‌دیده در فایل `trained_model.joblib`.


In [ ]:
MAX_LEN_F1, MAX_LEN_F2, MAX_LEN_F3 = 10, 20, 5
MIN_VAL_F1, MAX_VAL_F1 = 1000.0, 10000.0
MIN_VAL_F2, MAX_VAL_F2 = 20000.0, 100000.0
MIN_VAL_F3, MAX_VAL_F3 = 0.1, 1000.0

def encode_feature_vector(points: List[float], max_len: int, min_val: float, max_val: float) -> Tuple[List[float], List[float]]:
    if not points: return [0.0] * max_len, [0.0] * max_len
    sorted_pts = sorted(points)
    span = max_val - min_val
    norm_pts = [(x - min_val) / span for x in sorted_pts]
    actual_len = len(norm_pts)
    pad_len = max(0, max_len - actual_len)
    values = norm_pts[:max_len] + [0.0] * pad_len
    mask = [1.0] * min(actual_len, max_len) + [0.0] * pad_len
    return values, mask

def encode_variable_length_query(q1: List[float], q2: List[float], q3: List[float]) -> np.ndarray:
    v1, m1 = encode_feature_vector(q1, MAX_LEN_F1, MIN_VAL_F1, MAX_VAL_F1)
    v2, m2 = encode_feature_vector(q2, MAX_LEN_F2, MIN_VAL_F2, MAX_VAL_F2)
    v3, m3 = encode_feature_vector(q3, MAX_LEN_F3, MIN_VAL_F3, MAX_VAL_F3)
    return np.array(v1 + v2 + v3 + m1 + m2 + m3, dtype=np.float32)

def generate_training_data(df: pd.DataFrame, samples_per_class: int = 80):
    parsed = [(parse_intervals(r["Feature1"]), parse_intervals(r["Feature2"]), parse_intervals(r["Feature3"])) for _, r in df.iterrows()]
    X, y = [], []
    for row_idx in range(len(df)):
        d1, d2, d3 = parsed[row_idx]
        for _ in range(samples_per_class):
            q1, _ = generate_feature_points(intervals_to_string(d1))
            q2, _ = generate_feature_points(intervals_to_string(d2))
            q3, _ = generate_feature_points(intervals_to_string(d3))
            X.append(encode_variable_length_query(q1, q2, q3))
            y.append(row_idx)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

print("⏳ در حال تولید ۸۰۰۰ نمونه داده آموزشی (با حذف مستقل ۳۰٪ اعضا)...")
X, y = generate_training_data(df_database, samples_per_class=80)
print(f"✅ ماتریس ورودی X با ابعاد {X.shape} ساخته شد.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15
)

print("⏳ در حال آموزش شبکه عصبی عمیق روی ورودی‌های طول متغیر...")
model.fit(X_train, y_train)

# ذخیره وزن‌های مدل
joblib.dump(model, "trained_model.joblib")
print("💾 وزن‌های مدل با موفقیت در trained_model.joblib ذخیره شد.")

# ارزیابی دقت و رسم نمودار
train_acc = accuracy_score(y_train, model.predict(X_train)) * 100.0
test_acc = accuracy_score(y_test, model.predict(X_test)) * 100.0
test_probs = model.predict_proba(X_test)
top2_acc = np.mean([y_test[i] in np.argsort(test_probs[i])[-2:] for i in range(len(y_test))]) * 100.0
top3_acc = np.mean([y_test[i] in np.argsort(test_probs[i])[-3:] for i in range(len(y_test))]) * 100.0
top5_acc = np.mean([y_test[i] in np.argsort(test_probs[i])[-5:] for i in range(len(y_test))]) * 100.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(model.loss_curve_, color="#1f77b4", lw=2.5, label="Training Loss")
axes[0].set_title("Training Loss Convergence Curve", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Epochs / Iterations", fontsize=10)
axes[0].set_ylabel("Cross-Entropy Loss", fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

if model.validation_scores_:
    val_accs = [v * 100.0 for v in model.validation_scores_]
    axes[1].plot(val_accs, color="#2ca02c", lw=2.5, marker="o", markersize=4, label="Validation Accuracy")
    axes[1].set_title("Validation Accuracy over Epochs", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Epochs", fontsize=10)
    axes[1].set_ylabel("Accuracy (%)", fontsize=10)
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()

metrics = ["Train Acc", "Test Top-1", "Test Top-2", "Test Top-3", "Test Top-5"]
values = [train_acc, test_acc, top2_acc, top3_acc, top5_acc]
colors = ["#3498db", "#e74c3c", "#e67e22", "#2ecc71", "#1abc9c"]

bars = axes[2].bar(metrics, values, color=colors, width=0.55, edgecolor="black", linewidth=0.8)
axes[2].set_ylim(0, 115)
axes[2].set_title("Algorithm Performance: Train vs Unseen Test Data", fontsize=11, fontweight="bold")
axes[2].set_ylabel("Accuracy (%)", fontsize=10)
axes[2].grid(axis="y", linestyle="--", alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width() / 2.0, yval + 1.5, f"{yval:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"✅ دقت آموزش: {train_acc:.2f}% | دقت در تست Top-1: {test_acc:.2f}% | دقت در تست Top-3: {top3_acc:.2f}%")


## مرحله چهارم: استنتاج سریع با بارگذاری وزن‌های مدل (بدون ترین مجدد)

مطابق خواسته کارفرما:
«با توجه به اینکه ترین زمان‌بر است، یک کد جداگانه ارائه شود که وزن‌های مدل ترین‌شده را به صورت جداگانه دریافت کند و خروجی را بدهد.»
در این سلول وزن‌های ذخیره‌شده از `trained_model.joblib` بارگذاری شده و استنتاج در کسری از ثانیه اجرا می‌شود:


In [ ]:
# بارگذاری وزن‌های مدل آموزش‌دیده
loaded_model = joblib.load("trained_model.joblib")
print("⚡ وزن‌های مدل از trained_model.joblib با موفقیت لود شدند.")

# اجرای استنتاج مستقیم
x_input = encode_variable_length_query(
    query_sample["Feature1_values"],
    query_sample["Feature2_values"],
    query_sample["Feature3_values"]
)

probs = loaded_model.predict_proba([x_input])[0]
top3_indices = np.argsort(probs)[-3:][::-1]

results = []
for rank, idx in enumerate(top3_indices, 1):
    row = df_database.iloc[idx]
    results.append({
        "رتبه": f"🥇 رتبه {rank}",
        "شماره سطر": int(idx),
        "درصد احتمال شباهت": f"{probs[idx] * 100:.2f}%",
        "Feature1 بازه‌ای": row["Feature1"][:40] + "...",
        "Feature2 بازه‌ای": row["Feature2"][:40] + "...",
        "Feature3 بازه‌ای": row["Feature3"]
    })

pd.DataFrame(results)
